# The molecular model, offline

Build the **Rayleigh (molecular) reference profile** from the US Standard Atmosphere
1976 table shipped with the package — this runs with **no external data**, so it is a
good install sanity-check and a way to explore the molecular backscatter the
calibration fits against.

## 1. Load the shipped standard atmosphere

In [ ]:
import numpy as np
from calibration.rayleigh.atmosphere import (
    load_standard_atmosphere,
    calculate_molecular_properties,
    DEFAULT_STANDARD_ATMOSPHERE,
)

print("package data file:", DEFAULT_STANDARD_ATMOSPHERE)
print("exists           :", DEFAULT_STANDARD_ATMOSPHERE.exists())

station_alt = 491.0                          # Payerne, m ASL
range_agl = np.arange(0.0, 12000.0, 30.0)    # 30 m E-PROFILE grid
altitude_asl = station_alt + range_agl

# Passing None uses the package-shipped US Standard Atmosphere 1976
atm = load_standard_atmosphere(None, altitude_asl)
i4 = int(np.argmin(np.abs(range_agl - 4000)))
print("T surface / 4 km :", round(float(atm.temperature[0]), 1), "/",
      round(float(atm.temperature[i4]), 1), "K")

## 2. Molecular backscatter at 1064 nm vs 910 nm

In [ ]:
mol_1064 = calculate_molecular_properties(atm.temperature, atm.pressure, range_agl, 1064.47e-9)
mol_910  = calculate_molecular_properties(atm.temperature, atm.pressure, range_agl, 910.74e-9)

i2 = int(np.argmin(np.abs(range_agl - 2000)))
print(f"beta_mol @2 km  1064 nm : {mol_1064.beta_mol[i2]:.3e}  1/(m sr)")
print(f"beta_mol @2 km   910 nm : {mol_910.beta_mol[i2]:.3e}  1/(m sr)")
print(f"ratio 910 / 1064        : {mol_910.beta_mol[i2] / mol_1064.beta_mol[i2]:.3f}")

## 3. Optional: plot the profiles (landscape)

In [ ]:
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(mol_1064.beta_mol, altitude_asl / 1000, label="1064 nm")
    ax.plot(mol_910.beta_mol,  altitude_asl / 1000, label="910 nm")
    ax.set_xscale("log")
    ax.set_xlabel(r"$eta_{mol}$  [1/(m sr)]")
    ax.set_ylabel("Altitude (km ASL)")
    ax.set_title("Molecular backscatter — US Standard Atmosphere 1976")
    ax.legend(); ax.grid(alpha=0.3); fig.tight_layout()
except ModuleNotFoundError:
    print("matplotlib not installed - `pip install matplotlib` to see the plot.")

## 4. Available molecular-window detection methods

In [ ]:
from calibration.rayleigh.molecular_methods import METHODS

print("Selectable via options.json -> molecular_method:")
for m in METHODS:
    print("  -", m)

See `validation/compare_molecular_methods.py` for a full multi-method comparison on
real nights, and the reports under `doc/reports/` for the long-run results.